# Feature Attribution Analysis
## Integrated Gradients, GradientSHAP, and Attention Rollout on Full Validation Sets

This notebook computes token-level attributions using three methods:
1. **Integrated Gradients (IG)** — gradient-based, Captum
2. **GradientSHAP (SHAP)** — gradient-based, Captum
3. **Attention Rollout (AR)** — manual Q/K computation (SDPA workaround)

Run on the **full validation sets** of both PAN25 (3556 examples) and AuthorMix (3642 examples).

## §1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import nbformat

path_in = "/content/drive/MyDrive/ap-thesis/jupyter/attribution.ipynb"
path_out = "/content/drive/MyDrive/ap-thesis/attribution_github.ipynb"

nb = nbformat.read(path_in, as_version=4)

# remove broken top-level widget metadata
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

# remove widget-view outputs, keep normal text / tables / plots
for cell in nb.cells:
    if "outputs" in cell:
        cleaned = []
        for out in cell.outputs:
            data = out.get("data", {})
            if "application/vnd.jupyter.widget-view+json" in data:
                continue
            cleaned.append(out)
        cell.outputs = cleaned

nbformat.write(nb, path_out)
print(f"saved: {path_out}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install captum

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import numpy as np, pandas as pd
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

numpy: 1.26.4
pandas: 2.2.2


In [ ]:
import os, json, math, re, time
import numpy as np
import pandas as pd
import torch

from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import IntegratedGradients, GradientShap

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words("english"))

ROOT = "/content/drive/MyDrive/ap-thesis"
RUNS_ROOT = f"{ROOT}/runs"
os.makedirs(f"{RUNS_ROOT}/results", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


## §2 — Load Models & Validation Sets

In [ ]:
def load_best_seed(csv_path, f1_col="eval_f1_macro"):
    df = pd.read_csv(csv_path)
    assert f1_col in df.columns, f"Missing {f1_col} in {csv_path}"
    return int(df.sort_values(f1_col, ascending=False).iloc[0]["seed"])


# ── PAN25 multiclass ──
best_seed_pmc = load_best_seed(f"{RUNS_ROOT}/results/pan25_multiclass_seeds.csv")
best_dir_pmc  = f"{RUNS_ROOT}/models/pan25_multiclass_best_seed{best_seed_pmc}"
tok_dir_pmc   = f"{RUNS_ROOT}/cache/pan25_multiclass_tok512"

with open(f"{RUNS_ROOT}/models/pan25_multiclass_label_map.json") as f:
    label_map_pmc = {int(k): v for k, v in json.load(f).items()}

pan_tok   = AutoTokenizer.from_pretrained(best_dir_pmc, use_fast=True)
pan_model = AutoModelForSequenceClassification.from_pretrained(best_dir_pmc).to(DEVICE)
pan_val   = load_from_disk(tok_dir_pmc)["validation"]

print(f"PAN25 best model: {best_dir_pmc}")
print(f"PAN25 validation size: {len(pan_val)}")


# ── AuthorMix ──
best_seed_am = load_best_seed(f"{RUNS_ROOT}/results/authormix_seeds.csv")
best_dir_am  = f"{RUNS_ROOT}/models/authormix_best_seed{best_seed_am}"
tok_dir_am   = f"{RUNS_ROOT}/cache/authormix_tok512"

with open(f"{best_dir_am}/label_map.json") as f:
    label_map_am = {int(k): v for k, v in json.load(f).items()}

am_tok   = AutoTokenizer.from_pretrained(best_dir_am, use_fast=True)
am_model = AutoModelForSequenceClassification.from_pretrained(best_dir_am).to(DEVICE)
am_val   = load_from_disk(tok_dir_am)["validation"]

print(f"AuthorMix best model: {best_dir_am}")
print(f"AuthorMix validation size: {len(am_val)}")

PAN25 best model: /content/drive/MyDrive/ap-thesis/runs/models/pan25_multiclass_best_seed2022
PAN25 validation size: 3556


AuthorMix best model: /content/drive/MyDrive/ap-thesis/runs/models/authormix_best_seed11
AuthorMix validation size: 3642


## §2b — Balanced Accuracy (Mean +/- Std across Seeds) & Per-Class Accuracy

In [ ]:
# ── Balanced Accuracy across seeds ──

pan_seeds_df = pd.read_csv(f"{RUNS_ROOT}/results/pan25_multiclass_seeds.csv")
am_seeds_df  = pd.read_csv(f"{RUNS_ROOT}/results/authormix_seeds.csv")

pan_bal_acc = pan_seeds_df["eval_balanced_accuracy"]
am_bal_acc  = am_seeds_df["eval_balanced_accuracy"]

print(f"PAN25 multiclass balanced accuracy across {len(pan_bal_acc)} seeds:")
print(f"  Mean: {pan_bal_acc.mean():.4f}  Std: {pan_bal_acc.std():.4f}")
print(f"  Per seed: {pan_bal_acc.values}")
print()
print(f"AuthorMix balanced accuracy across {len(am_bal_acc)} seeds:")
print(f"  Mean: {am_bal_acc.mean():.4f}  Std: {am_bal_acc.std():.4f}")
print(f"  Per seed: {am_bal_acc.values}")

# Save summary
bal_acc_summary = pd.DataFrame([
    {"dataset": "PAN25 multiclass", "n_seeds": len(pan_bal_acc),
     "mean_balanced_accuracy": round(pan_bal_acc.mean(), 4),
     "std_balanced_accuracy": round(pan_bal_acc.std(), 4),
     "best_seed": best_seed_pmc},
    {"dataset": "AuthorMix", "n_seeds": len(am_bal_acc),
     "mean_balanced_accuracy": round(am_bal_acc.mean(), 4),
     "std_balanced_accuracy": round(am_bal_acc.std(), 4),
     "best_seed": best_seed_am},
])
bal_acc_path = f"{RUNS_ROOT}/results/balanced_accuracy_summary.csv"
bal_acc_summary.to_csv(bal_acc_path, index=False)
print(f"\nSaved: {bal_acc_path}")
display(bal_acc_summary)

PAN25 multiclass balanced accuracy across 3 seeds:
  Mean: 0.6152  Std: 0.0088
  Per seed: [0.62264615 0.61745915 0.60540341]

AuthorMix balanced accuracy across 3 seeds:
  Mean: 0.8572  Std: 0.0052
  Per seed: [0.86172704 0.85835514 0.85158643]

Saved: /content/drive/MyDrive/ap-thesis/runs/results/balanced_accuracy_summary.csv


,dataset,n_seeds,mean_balanced_accuracy,std_balanced_accuracy,best_seed
0,PAN25 multiclass,3,0.6152,0.0088,2022
1,AuthorMix,3,0.8572,0.0052,11


In [ ]:
# ── Per-Class Accuracy (best-seed model on validation set) ──

def compute_per_class_accuracy(model, dataset, label_map, batch_size=64):
    """
    Run inference on the full validation set and compute per-class accuracy.
    Returns a DataFrame with label_id, class_name, support, class_accuracy.
    """
    model.eval()
    all_preds = []
    all_labels = []

    for start in range(0, len(dataset), batch_size):
        end = min(start + batch_size, len(dataset))
        batch = dataset[start:end]

        # Pad ragged input_ids / attention_mask to the same length within batch
        ids_list  = batch["input_ids"]
        mask_list = batch["attention_mask"]
        max_len = max(len(seq) for seq in ids_list)

        padded_ids  = torch.zeros(len(ids_list), max_len, dtype=torch.long)
        padded_mask = torch.zeros(len(ids_list), max_len, dtype=torch.long)
        for j, (ids, mask) in enumerate(zip(ids_list, mask_list)):
            length = len(ids)
            padded_ids[j, :length]  = torch.tensor(ids, dtype=torch.long)
            padded_mask[j, :length] = torch.tensor(mask, dtype=torch.long)

        padded_ids  = padded_ids.to(DEVICE)
        padded_mask = padded_mask.to(DEVICE)
        labels = batch["label"]

        with torch.no_grad():
            logits = model(input_ids=padded_ids, attention_mask=padded_mask).logits
            preds = logits.argmax(dim=-1).cpu().tolist()

        all_preds.extend(preds)
        all_labels.extend(labels)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    rows = []
    for label_id in sorted(label_map.keys()):
        mask = all_labels == label_id
        support = int(mask.sum())
        if support == 0:
            continue
        correct = int((all_preds[mask] == label_id).sum())
        rows.append({
            "label_id": label_id,
            "class_name": label_map[label_id],
            "support": support,
            "class_accuracy": correct / support,
        })

    df = pd.DataFrame(rows).sort_values("class_accuracy", ascending=False)

    # Balanced accuracy = mean of per-class accuracies
    balanced_acc = df["class_accuracy"].mean()
    print(f"  Balanced accuracy (validation): {balanced_acc:.4f}")

    return df, balanced_acc


print("PAN25 multiclass — per-class accuracy (best seed):")
pan_pca, pan_ba = compute_per_class_accuracy(pan_model, pan_val, label_map_pmc)
pan_pca_path = f"{RUNS_ROOT}/results/pan25_multiclass_per_class_accuracy_validation.csv"
pan_pca.to_csv(pan_pca_path, index=False)
print(f"  Saved: {pan_pca_path}")
display(pan_pca)

print("\nAuthorMix — per-class accuracy (best seed):")
am_pca, am_ba = compute_per_class_accuracy(am_model, am_val, label_map_am)
am_pca_path = f"{RUNS_ROOT}/results/authormix_per_class_accuracy_validation.csv"
am_pca.to_csv(am_pca_path, index=False)
print(f"  Saved: {am_pca_path}")
display(am_pca)

PAN25 multiclass — per-class accuracy (best seed):
  Balanced accuracy (validation): 0.6800
  Saved: /content/drive/MyDrive/ap-thesis/runs/results/pan25_multiclass_per_class_accuracy_validation.csv


,label_id,class_name,support,class_accuracy
16,16,llama-3.3-70b-instruct,60,0.983333
0,0,deepseek-r1-distill-qwen-32b,135,0.940741
6,6,gpt-3.5-turbo,206,0.922330
12,12,human,1365,0.919414
7,7,gpt-4-turbo,41,0.902439
9,9,gpt-4.5-preview,41,0.853659
14,14,llama-2-7b-chat,40,0.850000
17,17,ministral-8b-instruct-2410,165,0.775758
1,1,falcon3-10b-instruct,132,0.772727
20,20,o3-mini,161,0.763975



AuthorMix — per-class accuracy (best seed):
  Balanced accuracy (validation): 0.9262
  Saved: /content/drive/MyDrive/ap-thesis/runs/results/authormix_per_class_accuracy_validation.csv


,label_id,class_name,support,class_accuracy
7,7,h,14,1.000000
10,10,pp,17,1.000000
11,11,qq,13,1.000000
0,0,blog11518,510,0.974510
3,3,blog30407,161,0.962733
8,8,hemingway,504,0.954365
4,4,blog5546,160,0.937500
6,6,fitzgerald,885,0.931073
13,13,woolf,488,0.928279
1,1,blog25872,60,0.900000


In [ ]:
def load_best_seed(csv_path, f1_col="eval_f1_macro"):
    df = pd.read_csv(csv_path)
    assert f1_col in df.columns, f"Missing {f1_col} in {csv_path}"
    return int(df.sort_values(f1_col, ascending=False).iloc[0]["seed"])


# ── PAN25 multiclass ──
best_seed_pmc = load_best_seed(f"{RUNS_ROOT}/results/pan25_multiclass_seeds.csv")
best_dir_pmc  = f"{RUNS_ROOT}/models/pan25_multiclass_best_seed{best_seed_pmc}"
tok_dir_pmc   = f"{RUNS_ROOT}/cache/pan25_multiclass_tok512"

with open(f"{RUNS_ROOT}/models/pan25_multiclass_label_map.json") as f:
    label_map_pmc = {int(k): v for k, v in json.load(f).items()}

pan_tok   = AutoTokenizer.from_pretrained(best_dir_pmc, use_fast=True)
pan_model = AutoModelForSequenceClassification.from_pretrained(best_dir_pmc).to(DEVICE)
pan_val   = load_from_disk(tok_dir_pmc)["validation"]

print(f"PAN25 best model: {best_dir_pmc}")
print(f"PAN25 validation size: {len(pan_val)}")


# ── AuthorMix ──
best_seed_am = load_best_seed(f"{RUNS_ROOT}/results/authormix_seeds.csv")
best_dir_am  = f"{RUNS_ROOT}/models/authormix_best_seed{best_seed_am}"
tok_dir_am   = f"{RUNS_ROOT}/cache/authormix_tok512"

with open(f"{best_dir_am}/label_map.json") as f:
    label_map_am = {int(k): v for k, v in json.load(f).items()}

am_tok   = AutoTokenizer.from_pretrained(best_dir_am, use_fast=True)
am_model = AutoModelForSequenceClassification.from_pretrained(best_dir_am).to(DEVICE)
am_val   = load_from_disk(tok_dir_am)["validation"]

print(f"AuthorMix best model: {best_dir_am}")
print(f"AuthorMix validation size: {len(am_val)}")

PAN25 best model: /content/drive/MyDrive/ap-thesis/runs/models/pan25_multiclass_best_seed2022
PAN25 validation size: 3556


AuthorMix best model: /content/drive/MyDrive/ap-thesis/runs/models/authormix_best_seed11
AuthorMix validation size: 3642


## §3 — Core Attribution Functions

In [ ]:
def decode_tokens(tokenizer, input_ids_1d):
    """Convert token IDs to string tokens."""
    return tokenizer.convert_ids_to_tokens(input_ids_1d.tolist())


def summarize_attr(attr_3d):
    """Reduce [1, seq, hidden] attribution to [seq] via abs-sum over hidden dim."""
    return attr_3d.squeeze(0).abs().sum(dim=-1).detach().cpu().numpy()


def top_tokens(tokens, scores, k=15):
    """Return top-k (token, score) pairs, excluding special tokens."""
    pairs = [(t, float(s)) for t, s in zip(tokens, scores)
             if t not in ("[CLS]", "[SEP]", "[PAD]")]
    pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return pairs[:k]


def is_punctuation(tok):
    return bool(re.fullmatch(r"\W+", tok))


def is_stopword(tok):
    return tok.lower() in STOP_WORDS

In [ ]:
# /// Attention Rollout

def attention_rollout_bert_manual(model, tokenizer, ex, start_layer=0):
    """
    Attention Rollout for BERT that does NOT use output_attentions=True.
    Computes attention matrices from Q/K weights directly.
    Returns CLS->token rollout scores: np.array shape [seq_len]

    Because SDPA did not expose attention weights in our environment,
    we compute per-layer attention probabilities from the model's Q/K
    projections and apply standard attention rollout.
    """
    model.eval()

    input_ids = torch.tensor([ex["input_ids"]], dtype=torch.long).to(DEVICE)
    attention_mask = torch.tensor([ex["attention_mask"]], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        out = model.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )
    hidden_states = out.hidden_states

    B, T = input_ids.shape
    device = input_ids.device

    attn_mask = attention_mask[:, None, None, :].to(dtype=torch.float32)
    additive_mask = (1.0 - attn_mask) * -1e4

    attn_per_layer = []
    num_layers = len(model.bert.encoder.layer)

    for layer_idx in range(num_layers):
        layer = model.bert.encoder.layer[layer_idx]
        sa = layer.attention.self

        h = hidden_states[layer_idx]
        q = sa.query(h)
        k = sa.key(h)

        num_heads = sa.num_attention_heads
        head_dim = sa.attention_head_size

        def shape(x):
            return x.view(B, T, num_heads, head_dim).permute(0, 2, 1, 3)

        q = shape(q)
        k = shape(k)

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(head_dim)
        scores = scores + additive_mask
        probs = torch.softmax(scores, dim=-1)

        probs_mean = probs.mean(dim=1).squeeze(0)  # [T, T]
        attn_per_layer.append(probs_mean)

    # Rollout: add residual + normalize + multiply across layers
    eye = torch.eye(T, device=device)
    attn_per_layer = [(a + eye) / (a + eye).sum(dim=-1, keepdim=True)
                      for a in attn_per_layer]

    joint = attn_per_layer[start_layer]
    for i in range(start_layer + 1, len(attn_per_layer)):
        joint = attn_per_layer[i] @ joint

    cls_scores = joint[0].detach().cpu().numpy()
    return cls_scores

In [ ]:
# ── Single-Example Attribution (all 3 methods) ──

def explain_one(model, tokenizer, ex, ig_steps=25, gs_samples=10, gs_stdevs=0.09):
    """
    Run IG + GradientShap + Attention Rollout on a single example.
    Returns dict with tokens and top-k for each method.
    """
    model.eval()

    input_ids = torch.tensor([ex["input_ids"]], dtype=torch.long).to(DEVICE)
    attention_mask = torch.tensor([ex["attention_mask"]], dtype=torch.long).to(DEVICE)
    target = int(ex["label"])

    emb_layer = model.bert.embeddings
    inputs_embeds = emb_layer(input_ids)

    pad_id = tokenizer.pad_token_id
    baseline_ids = torch.full_like(input_ids, fill_value=pad_id, dtype=torch.long)
    baseline_embeds = emb_layer(baseline_ids)

    def forward_embeds(embeds):
        return model(inputs_embeds=embeds, attention_mask=attention_mask).logits

    # Integrated Gradients
    ig = IntegratedGradients(forward_embeds)
    ig_attr = ig.attribute(
        inputs=inputs_embeds, baselines=baseline_embeds,
        target=target, n_steps=ig_steps
    )

    # GradientShap
    gs = GradientShap(forward_embeds)
    gs_attr = gs.attribute(
        inputs=inputs_embeds, baselines=baseline_embeds,
        target=target, n_samples=gs_samples, stdevs=gs_stdevs
    )

    # Attention Rollout
    ar_scores = attention_rollout_bert_manual(model, tokenizer, ex, start_layer=0)

    toks = decode_tokens(tokenizer, input_ids.squeeze(0))
    ig_scores = summarize_attr(ig_attr)
    gs_scores = summarize_attr(gs_attr)

    return {
        "true_label": target,
        "tokens": toks,
        "top_ig": top_tokens(toks, ig_scores, k=15),
        "top_gs": top_tokens(toks, gs_scores, k=15),
        "top_ar": top_tokens(toks, ar_scores, k=15),
    }

## §4 — Full Validation Attribution (All 3 Methods)

Runs IG, GradientSHAP, and Attention Rollout on every example in the validation set.
Saves per-example top-15 tokens + content/punct/stop ratios for each method.

In [ ]:
def compute_token_ratios(token_list):
    """Compute content/punct/stop ratios from a list of token strings."""
    if not token_list:
        return 0.0, 0.0, 0.0
    punct = sum(is_punctuation(t) for t in token_list)
    stop  = sum(is_stopword(t) for t in token_list)
    content = len(token_list) - punct - stop
    n = len(token_list)
    return content / n, punct / n, stop / n


def full_validation_attribution(
    model, tokenizer, dataset, label_map,
    ig_steps=15, gs_samples=10, gs_stdevs=0.09, top_k=15,
    save_csv=None, save_json=None,
    log_every=100,
):
    """
    Run all 3 attribution methods (IG, GradientSHAP, AR) on the full dataset.

    Returns:
        ratios_df: DataFrame with per-example ratios for each method
        all_expls: list of dicts with full token-level attributions
    """
    model.eval()
    emb_layer = model.bert.embeddings
    pad_id = tokenizer.pad_token_id

    ratio_rows = []
    all_expls = []
    t0 = time.time()

    for i in range(len(dataset)):
        ex = dataset[i]
        input_ids = torch.tensor([ex["input_ids"]], dtype=torch.long).to(DEVICE)
        attention_mask = torch.tensor([ex["attention_mask"]], dtype=torch.long).to(DEVICE)
        target = int(ex["label"])
        class_name = label_map[target]

        inputs_embeds = emb_layer(input_ids)
        baseline_ids = torch.full_like(input_ids, fill_value=pad_id, dtype=torch.long)
        baseline_embeds = emb_layer(baseline_ids)

        def forward_embeds(embeds):
            return model(inputs_embeds=embeds, attention_mask=attention_mask).logits

        # ── IG ──
        ig = IntegratedGradients(forward_embeds)
        ig_attr = ig.attribute(
            inputs=inputs_embeds, baselines=baseline_embeds,
            target=target, n_steps=ig_steps
        )

        # ── GradientSHAP ──
        gs = GradientShap(forward_embeds)
        gs_attr = gs.attribute(
            inputs=inputs_embeds, baselines=baseline_embeds,
            target=target, n_samples=gs_samples, stdevs=gs_stdevs
        )

        # ── Attention Rollout ──
        ar_scores = attention_rollout_bert_manual(model, tokenizer, ex, start_layer=0)

        toks = decode_tokens(tokenizer, input_ids.squeeze(0))
        ig_scores = summarize_attr(ig_attr)
        gs_scores = summarize_attr(gs_attr)

        top_ig = top_tokens(toks, ig_scores, k=top_k)
        top_gs = top_tokens(toks, gs_scores, k=top_k)
        top_ar = top_tokens(toks, ar_scores, k=top_k)

        # Token ratios per method
        ig_c, ig_p, ig_s = compute_token_ratios([t[0] for t in top_ig])
        gs_c, gs_p, gs_s = compute_token_ratios([t[0] for t in top_gs])
        ar_c, ar_p, ar_s = compute_token_ratios([t[0] for t in top_ar])

        ratio_rows.append({
            "example_idx": i,
            "true_label": target,
            "class": class_name,
            # IG ratios
            "ig_content_ratio": ig_c, "ig_punct_ratio": ig_p, "ig_stop_ratio": ig_s,
            # SHAP ratios
            "gs_content_ratio": gs_c, "gs_punct_ratio": gs_p, "gs_stop_ratio": gs_s,
            # AR ratios
            "ar_content_ratio": ar_c, "ar_punct_ratio": ar_p, "ar_stop_ratio": ar_s,
        })

        all_expls.append({
            "example_idx": i,
            "true_label": target,
            "class_name": class_name,
            "tokens": toks,
            "top_ig": top_ig,
            "top_gs": top_gs,
            "top_ar": top_ar,
        })

        if (i + 1) % log_every == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (len(dataset) - i - 1) / rate
            print(f"  [{i+1}/{len(dataset)}] {rate:.1f} ex/s | ETA {eta/60:.1f} min")

    ratios_df = pd.DataFrame(ratio_rows)

    if save_csv:
        ratios_df.to_csv(save_csv, index=False)
        print(f"Saved ratios CSV: {save_csv}")

    if save_json:
        with open(save_json, "w") as f:
            json.dump(all_expls, f)
        print(f"Saved full attributions JSON: {save_json}")

    elapsed = time.time() - t0
    print(f"Done: {len(dataset)} examples in {elapsed/60:.1f} min")

    return ratios_df, all_expls

### §4a — PAN25 Full Validation (all 3 methods)

In [ ]:
pan_ratios, pan_expls = full_validation_attribution(
    model=pan_model,
    tokenizer=pan_tok,
    dataset=pan_val,
    label_map=label_map_pmc,
    ig_steps=15,
    gs_samples=10,
    gs_stdevs=0.09,
    save_csv=f"{RUNS_ROOT}/results/pan25_full_validation_all_methods_ratios.csv",
    save_json=f"{RUNS_ROOT}/results/pan25_full_validation_all_methods_attributions.json",
)

  [100/3556] 0.6 ex/s | ETA 97.1 min
  [200/3556] 0.6 ex/s | ETA 95.4 min
  [300/3556] 0.6 ex/s | ETA 93.2 min
  [400/3556] 0.6 ex/s | ETA 90.5 min
  [500/3556] 0.6 ex/s | ETA 87.5 min
  [600/3556] 0.6 ex/s | ETA 84.7 min
  [700/3556] 0.6 ex/s | ETA 81.9 min
  [800/3556] 0.6 ex/s | ETA 79.2 min
  [900/3556] 0.6 ex/s | ETA 76.3 min
  [1000/3556] 0.6 ex/s | ETA 73.4 min
  [1100/3556] 0.6 ex/s | ETA 70.5 min
  [1200/3556] 0.6 ex/s | ETA 67.5 min
  [1300/3556] 0.6 ex/s | ETA 64.6 min
  [1400/3556] 0.6 ex/s | ETA 61.8 min
  [1500/3556] 0.6 ex/s | ETA 58.9 min
  [1600/3556] 0.6 ex/s | ETA 56.0 min
  [1700/3556] 0.6 ex/s | ETA 53.2 min
  [1800/3556] 0.6 ex/s | ETA 50.3 min
  [1900/3556] 0.6 ex/s | ETA 47.5 min
  [2000/3556] 0.6 ex/s | ETA 44.6 min
  [2100/3556] 0.6 ex/s | ETA 41.7 min
  [2200/3556] 0.6 ex/s | ETA 38.9 min
  [2300/3556] 0.6 ex/s | ETA 36.0 min
  [2400/3556] 0.6 ex/s | ETA 33.1 min
  [2500/3556] 0.6 ex/s | ETA 30.3 min
  [2600/3556] 0.6 ex/s | ETA 27.4 min
  [2700/3556] 0.6 ex/

### §4b — AuthorMix Full Validation (all 3 methods)

In [ ]:
am_ratios, am_expls = full_validation_attribution(
    model=am_model,
    tokenizer=am_tok,
    dataset=am_val,
    label_map=label_map_am,
    ig_steps=15,
    gs_samples=10,
    gs_stdevs=0.09,
    save_csv=f"{RUNS_ROOT}/results/authormix_full_validation_all_methods_ratios.csv",
    save_json=f"{RUNS_ROOT}/results/authormix_full_validation_all_methods_attributions.json",
)

  [100/3642] 3.8 ex/s | ETA 15.5 min
  [200/3642] 3.6 ex/s | ETA 15.8 min
  [300/3642] 3.6 ex/s | ETA 15.3 min
  [400/3642] 3.7 ex/s | ETA 14.4 min
  [500/3642] 3.9 ex/s | ETA 13.3 min
  [600/3642] 4.1 ex/s | ETA 12.4 min
  [700/3642] 4.2 ex/s | ETA 11.6 min
  [800/3642] 4.2 ex/s | ETA 11.4 min
  [900/3642] 4.0 ex/s | ETA 11.3 min
  [1000/3642] 4.0 ex/s | ETA 11.1 min
  [1100/3642] 3.9 ex/s | ETA 10.8 min
  [1200/3642] 3.8 ex/s | ETA 10.6 min
  [1300/3642] 3.9 ex/s | ETA 10.1 min
  [1400/3642] 3.9 ex/s | ETA 9.6 min
  [1500/3642] 3.9 ex/s | ETA 9.1 min
  [1600/3642] 3.9 ex/s | ETA 8.6 min
  [1700/3642] 4.0 ex/s | ETA 8.2 min
  [1800/3642] 4.0 ex/s | ETA 7.7 min
  [1900/3642] 4.0 ex/s | ETA 7.3 min
  [2000/3642] 4.0 ex/s | ETA 6.8 min
  [2100/3642] 4.0 ex/s | ETA 6.4 min
  [2200/3642] 4.1 ex/s | ETA 5.9 min
  [2300/3642] 4.1 ex/s | ETA 5.4 min
  [2400/3642] 4.2 ex/s | ETA 5.0 min
  [2500/3642] 4.2 ex/s | ETA 4.5 min
  [2600/3642] 4.3 ex/s | ETA 4.1 min
  [2700/3642] 4.3 ex/s | ETA 3.7 m

## §5 — Quick Sanity Check

In [ ]:
print(f"PAN25 ratios shape: {pan_ratios.shape}")
print(f"AuthorMix ratios shape: {am_ratios.shape}")
print()

print("PAN25 — mean ratios by class (IG):")
display(pan_ratios.groupby("class")[["ig_content_ratio", "ig_punct_ratio", "ig_stop_ratio"]].mean().sort_values("ig_content_ratio", ascending=False))
print()

print("AuthorMix — mean ratios by class (IG):")
display(am_ratios.groupby("class")[["ig_content_ratio", "ig_punct_ratio", "ig_stop_ratio"]].mean().sort_values("ig_content_ratio", ascending=False))

PAN25 ratios shape: (3556, 12)
AuthorMix ratios shape: (3642, 12)

PAN25 — mean ratios by class (IG):


,ig_content_ratio,ig_punct_ratio,ig_stop_ratio
class,,,
mixtral-8x7b-instruct-v0.1,0.383333,0.320000,0.296667
gpt-4.5-preview,0.380488,0.248780,0.370732
gemini-pro,0.370732,0.330081,0.299187
mistral-7b-instruct-v0.2,0.360000,0.361667,0.278333
gemini-pro-paraphrase,0.346667,0.343333,0.310000
qwen1.5-72b-chat-8bit,0.330081,0.365854,0.304065
gpt-4-turbo-paraphrase,0.323810,0.326984,0.349206
llama-2-70b-chat,0.318699,0.351220,0.330081
llama-2-7b-chat,0.316667,0.355000,0.328333



AuthorMix — mean ratios by class (IG):


,ig_content_ratio,ig_punct_ratio,ig_stop_ratio
class,,,
bush,0.594519,0.134395,0.271086
h,0.566667,0.142857,0.290476
woolf,0.558880,0.190984,0.250137
blog5546,0.551142,0.210422,0.238436
qq,0.528205,0.158974,0.312821
pp,0.525490,0.156863,0.317647
trump,0.519931,0.160320,0.319749
obama,0.497558,0.167574,0.334868
hemingway,0.471958,0.243254,0.284788


In [ ]:
print(f"PAN25 attributions saved: {len(pan_expls)} examples")
print(f"AuthorMix attributions saved: {len(am_expls)} examples")
print()
print("Files saved:")
print(f"  {RUNS_ROOT}/results/pan25_full_validation_all_methods_ratios.csv")
print(f"  {RUNS_ROOT}/results/pan25_full_validation_all_methods_attributions.json")
print(f"  {RUNS_ROOT}/results/authormix_full_validation_all_methods_ratios.csv")
print(f"  {RUNS_ROOT}/results/authormix_full_validation_all_methods_attributions.json")

PAN25 attributions saved: 3556 examples
AuthorMix attributions saved: 3642 examples

Files saved:
  /content/drive/MyDrive/ap-thesis/runs/results/pan25_full_validation_all_methods_ratios.csv
  /content/drive/MyDrive/ap-thesis/runs/results/pan25_full_validation_all_methods_attributions.json
  /content/drive/MyDrive/ap-thesis/runs/results/authormix_full_validation_all_methods_ratios.csv
  /content/drive/MyDrive/ap-thesis/runs/results/authormix_full_validation_all_methods_attributions.json
